# Flash Attention 3 教程

本教程介绍 Flash Attention 系列算法的原理和实现。

## 目录
1. 标准注意力的问题
2. Online Softmax 算法
3. Flash Attention V1/V2/V3
4. FP8 量化与 Incoherent Processing
5. 性能对比

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, 'src')

from flash_attn import (
    FlashAttentionConfig,
    OnlineSoftmax,
    FlashAttentionV1,
    FlashAttentionV2,
    FlashAttentionV3,
    FP8Quantizer,
    IncoherentProcessor,
    standard_attention,
    create_flash_attention,
    compute_attention_flops,
)

np.random.seed(42)
print("Flash Attention 模块加载成功!")

## 1. 标准注意力的问题

标准注意力计算：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right) V$$

**问题**：需要存储 $N \times N$ 的注意力矩阵，内存复杂度 $O(N^2)$

In [ ]:
# 内存占用分析
def memory_analysis(seq_lengths, head_dim=64, dtype_bytes=4):
    """分析不同序列长度的内存占用"""
    results = []
    for N in seq_lengths:
        # 标准注意力: Q, K, V + S + P
        standard = (3 * N * head_dim + 2 * N * N) * dtype_bytes / 1e9
        # Flash Attention: Q, K, V + O + l + m
        flash = (4 * N * head_dim + 2 * N) * dtype_bytes / 1e9
        results.append((N, standard, flash))
    return results

seq_lengths = [512, 1024, 2048, 4096, 8192, 16384]
memory = memory_analysis(seq_lengths)

plt.figure(figsize=(10, 5))
plt.plot([m[0] for m in memory], [m[1] for m in memory], 'r-o', label='Standard Attention')
plt.plot([m[0] for m in memory], [m[2] for m in memory], 'b-o', label='Flash Attention')
plt.xlabel('Sequence Length')
plt.ylabel('Memory (GB)')
plt.title('Memory Usage: Standard vs Flash Attention')
plt.legend()
plt.yscale('log')
plt.grid(True)
plt.show()

print("\n内存占用对比 (GB):")
print(f"{'Seq Len':<10} {'Standard':<15} {'Flash':<15} {'Ratio':<10}")
for N, std, flash in memory:
    print(f"{N:<10} {std:<15.4f} {flash:<15.6f} {std/flash:<10.1f}x")

## 2. Online Softmax 算法

Flash Attention 的核心是 **Online Softmax**，可以分块计算 softmax 而无需预先知道全局最大值。

### 算法原理

维护三个状态：
- $m$: 当前最大值
- $l$: 指数和
- $o$: 输出累积

每处理一个新块时更新这些状态。

In [ ]:
# 演示 Online Softmax
def demo_online_softmax():
    """演示在线 softmax 与标准 softmax 的等价性"""
    batch, seq_q, seq_kv, dim = 1, 4, 8, 16
    
    # 生成数据
    scores = np.random.randn(batch, seq_q, seq_kv).astype(np.float32)
    values = np.random.randn(batch, seq_kv, dim).astype(np.float32)
    
    # 标准 softmax
    scores_max = np.max(scores, axis=-1, keepdims=True)
    scores_exp = np.exp(scores - scores_max)
    weights = scores_exp / np.sum(scores_exp, axis=-1, keepdims=True)
    standard_output = np.einsum("bqk,bkd->bqd", weights, values)
    
    # Online softmax (分两块)
    softmax = OnlineSoftmax()
    state = softmax.init_state(batch, seq_q, dim)
    
    block_size = seq_kv // 2
    state = softmax.update(state, scores[:, :, :block_size], values[:, :block_size, :])
    state = softmax.update(state, scores[:, :, block_size:], values[:, block_size:, :])
    online_output = softmax.finalize(state)
    
    # 比较
    error = np.abs(standard_output - online_output).max()
    print(f"Standard Softmax Output Shape: {standard_output.shape}")
    print(f"Online Softmax Output Shape: {online_output.shape}")
    print(f"Max Absolute Error: {error:.2e}")
    print(f"Results Match: {error < 1e-5}")

demo_online_softmax()

## 3. Flash Attention V1/V2/V3

### 版本对比

| 版本 | 主要特性 | 性能提升 |
|:-----|:---------|:---------|
| V1 | 分块计算 + 重计算 | 2-4x |
| V2 | 优化并行 + 减少非 GEMM | 2x over V1 |
| V3 | Warp-Specialization + FP8 | 1.5-2x over V2 |

In [ ]:
# 创建测试数据
batch, seq, dim = 2, 128, 64
query = np.random.randn(batch, seq, dim).astype(np.float32)
key = np.random.randn(batch, seq, dim).astype(np.float32)
value = np.random.randn(batch, seq, dim).astype(np.float32)

# 标准注意力
std_output = standard_attention(query, key, value)

# Flash Attention 各版本
v1 = FlashAttentionV1()
v2 = FlashAttentionV2()
v3 = FlashAttentionV3()

v1_output = v1(query, key, value)
v2_output = v2(query, key, value)
v3_output = v3(query, key, value)

# 验证正确性
print("与标准注意力的误差:")
print(f"  V1: {np.abs(v1_output - std_output).max():.2e}")
print(f"  V2: {np.abs(v2_output - std_output).max():.2e}")
print(f"  V3: {np.abs(v3_output - std_output).max():.2e}")

In [ ]:
# 因果掩码测试
config = FlashAttentionConfig(causal=True, block_size_q=32, block_size_kv=32)
v3_causal = FlashAttentionV3(config)

causal_output = v3_causal(query, key, value)
std_causal = standard_attention(query, key, value, causal=True)

print(f"因果掩码误差: {np.abs(causal_output - std_causal).max():.2e}")

## 4. FP8 量化与 Incoherent Processing

Flash Attention V3 支持 FP8 低精度计算，可以进一步提升性能。

### FP8 格式
- **E4M3**: 4位指数 + 3位尾数，范围 ±448，精度更高
- **E5M2**: 5位指数 + 2位尾数，范围 ±57344，动态范围更大

In [ ]:
# FP8 量化演示
x = np.random.randn(256).astype(np.float32)

quantizer_e4m3 = FP8Quantizer(block_size=64, format="e4m3")
quantizer_e5m2 = FP8Quantizer(block_size=64, format="e5m2")

err_e4m3 = quantizer_e4m3.compute_quantization_error(x)
err_e5m2 = quantizer_e5m2.compute_quantization_error(x)

print("FP8 量化误差分析:")
print(f"\nE4M3 格式:")
print(f"  MSE: {err_e4m3['mse']:.6f}")
print(f"  Max Error: {err_e4m3['max_error']:.6f}")
print(f"  Relative Error: {err_e4m3['relative_error']:.4%}")

print(f"\nE5M2 格式:")
print(f"  MSE: {err_e5m2['mse']:.6f}")
print(f"  Max Error: {err_e5m2['max_error']:.6f}")
print(f"  Relative Error: {err_e5m2['relative_error']:.4%}")

In [ ]:
# Incoherent Processing 演示
# 创建有 outlier 的数据
x = np.random.randn(64, 64).astype(np.float32)
x[0, 0] = 50.0  # 添加 outlier

quantizer = FP8Quantizer(block_size=64)
processor = IncoherentProcessor(dim=64)

# 直接量化
x_direct, _ = quantizer.quantize(x)
error_direct = np.mean((x - x_direct) ** 2)

# Incoherent processing
x_incoherent = processor.process(x, quantizer)
error_incoherent = np.mean((x - x_incoherent) ** 2)

print("Incoherent Processing 效果:")
print(f"  直接量化 MSE: {error_direct:.6f}")
print(f"  Incoherent MSE: {error_incoherent:.6f}")
print(f"  误差降低: {(1 - error_incoherent/error_direct)*100:.1f}%")

## 5. 性能分析

In [ ]:
# FLOPs 分析
seq_lengths = [512, 1024, 2048, 4096]

print("注意力 FLOPs 分析 (batch=1, heads=8, dim=64):")
print(f"{'Seq Len':<10} {'Full (GFLOPs)':<15} {'Causal (GFLOPs)':<18} {'Savings':<10}")

for seq in seq_lengths:
    flops_full = compute_attention_flops(1, 8, seq, seq, 64, causal=False)
    flops_causal = compute_attention_flops(1, 8, seq, seq, 64, causal=True)
    savings = (1 - flops_causal['total'] / flops_full['total']) * 100
    print(f"{seq:<10} {flops_full['total']/1e9:<15.2f} {flops_causal['total']/1e9:<18.2f} {savings:<10.1f}%")

In [ ]:
# V3 调度器统计
v3 = FlashAttentionV3()
_ = v3(query, key, value)
stats = v3.get_scheduler_stats()

print("Flash Attention V3 调度器统计:")
for key, val in stats.items():
    print(f"  {key}: {val}")

## 6. 总结

Flash Attention 通过以下技术实现高效注意力计算：

1. **分块计算 (Tiling)**: 将 O(N²) 内存降至 O(N)
2. **Online Softmax**: 无需存储完整注意力矩阵
3. **IO 感知**: 最小化 HBM 访问
4. **Warp-Specialization (V3)**: 数据加载与计算并行
5. **FP8 量化 (V3)**: 进一步提升吞吐量

### 参考文献
- [Flash Attention V1](https://arxiv.org/abs/2205.14135)
- [Flash Attention V2](https://arxiv.org/abs/2307.08691)
- [Flash Attention V3](https://arxiv.org/abs/2407.08608)